Step 1 : 환경 설정 및 파일 다운로드

In [1]:
# 1. 환경 설정
!pip install pyspark wget

import wget
import csv
from pyspark.sql import SparkSession

# 2. Spark 초기화
spark = SparkSession.builder.master("local[*]").appName("HW1_Q3").getOrCreate()
sc = spark.sparkContext

# 3. 2015~2019년 CSV 파일 다운로드
base_url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World Hapiness Report/{}.csv"
years = [2015, 2016, 2017, 2018, 2019]

for year in years:
    wget.download(base_url.format(year), f"{year}.csv")

Step 2 : 데이터 파싱 및 병합

In [2]:
# 1. 데이터를 파싱하여 (국가명, (연도, 행복지수)) 형태로 변환하는 함수
def process_yearly_data(year):
    rdd = sc.textFile(f"{year}.csv")
    header = rdd.first()

    def extract_data(line):
        # csv 모듈을 사용하여 쉼표가 포함된 문자열을 안전하게 분리
        cols = next(csv.reader([line]))
        country = cols[0]                # 1번째 컬럼: Country
        score = round(float(cols[2]), 3) # 3번째 컬럼: Happiness Score
        return (country, (year, score))

    # 헤더 라인을 제외하고 데이터 추출
    return rdd.filter(lambda line: line != header).map(extract_data)

# 2. 연도별 RDD 생성
rdd_2015 = process_yearly_data(2015)
rdd_2016 = process_yearly_data(2016)
rdd_2017 = process_yearly_data(2017)
rdd_2018 = process_yearly_data(2018)
rdd_2019 = process_yearly_data(2019)

# 3. 5개의 RDD를 하나로 병합 (union 활용)
rdd_all_years = rdd_2015.union(rdd_2016).union(rdd_2017).union(rdd_2018).union(rdd_2019)

Step 3 : 그룹화 및 필터링

In [3]:
# 1. 국가를 기준으로 데이터 그룹화
rdd_grouped = rdd_all_years.groupByKey().mapValues(list)

# 2. 5년 연속 행복지수가 유지되거나 상승한 국가를 판별하는 함수
def is_consistently_increasing(year_score_list):
    # 5년 내내 데이터가 존재하는지 확인
    if len(year_score_list) != 5:
        return False

    # 연도를 기준으로 오름차순 정렬
    sorted_scores = sorted(year_score_list, key=lambda x: x[0])

    # 전년도 대비 지수가 하락한 적이 있는지 검사
    for i in range(1, 5):
        if sorted_scores[i][1] < sorted_scores[i-1][1]:
            return False
    return True

# 3. 조건에 맞는 국가만 필터링하여 국가명 추출
final_countries_rdd = rdd_grouped.filter(lambda x: is_consistently_increasing(x[1])).keys()

Step 4 : 최종 결과 수집 및 출력

In [4]:
# Action 연산을 통해 최종 결과 수집
happy_countries = final_countries_rdd.collect()

# 결과 출력
print("5년 동안 행복지수가 지속적으로 상승하거나 유지된 국가")
for country in sorted(happy_countries):
    print(f"- {country}")

5년 동안 행복지수가 지속적으로 상승하거나 유지된 국가
- Benin
- Burkina Faso
- Burundi
- Cambodia
- Cameroon
- Chad
- Congo (Brazzaville)
- Czech Republic
- Dominican Republic
- Estonia
- Finland
- Gabon
- Honduras
- Hungary
- Ivory Coast
- Latvia
- Malta
- Mongolia
- Niger
- Philippines
- Poland
- Portugal
- Romania
- Senegal
- Serbia
- Slovakia
- Syria
- Tajikistan
- Togo
